<a href="https://colab.research.google.com/github/GeomaticsCaminosUPM/seismicbuildingexposure/blob/main/seismicbuildingexposure/examples/footprint/position.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Position module example

This module computes the relative position of buildings

Install libraries if using colab

In [ ]:
!pip install geopandas
!pip install "seismicbuildingexposure @ git+https://github.com/GeomaticsCaminosUPM/seismic-building-exposure.git"
!pip install folium matplotlib mapclassify 

  Cloning https://github.com/GeomaticsCaminosUPM/SeismicBuildingExposure.git to /tmp/pip-install-dw6b7dn3/seismicbuildingexposure_b4de2aa630bf47e78d6f85b881a295dc
  Running command git clone --filter=blob:none --quiet https://github.com/GeomaticsCaminosUPM/SeismicBuildingExposure.git /tmp/pip-install-dw6b7dn3/seismicbuildingexposure_b4de2aa630bf47e78d6f85b881a295dc
  Resolved https://github.com/GeomaticsCaminosUPM/SeismicBuildingExposure.git to commit c68ac4a1701eeb7d1b7c07f86055c243056819a8
  Preparing metadata (setup.py) ... done
  Created wheel for SeismicBuildingExposure: filename=SeismicBuildingExposure-0.1.0-py3-none-any.whl size=22817 sha256=6fc93f97cdea8a6ef266c31a91f32146d9dfea98b07199401d771a26b6f4a05d
  Stored in directory: /tmp/pip-ephem-wheel-cache-ay8n2irw/wheels/f5/63/dc/048b0bbbfa53d621450952d08b42ee655e26308894e461ed4a
Successfully built SeismicBuildingExposure
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 1.7 MB/s eta 0:00:00


In [1]:
import seismicbuildingexposure.footprint
import geopandas as gpd

Documentation of every function can be found using `help()` or under https://github.com/GeomaticsCaminosUPM/SeismicBuildingExposure

In [2]:
help(seismicbuildingexposure.footprint.position.contact_forces_df)

Help on function contact_forces_df in module seismicbuildingexposure.footprint.position:

contact_forces_df(geoms: geopandas.geodataframe.GeoDataFrame, buffer: float = 0, height_column: str = None, min_radius: float = 0) -> pandas.core.frame.DataFrame
    Calculates force-based metrics for building footprints based on their geometry and proximity.

    Parameters:
        geoms (gpd.GeoDataFrame): A GeoDataFrame containing building footprints as polygon geometries.
        buffer (float): Buffer distance in meters to determine if two buildings are considered touching.
        height_column (str, optional): Column name in `geoms` specifying the building height in meters.
                                       If `None`, all buildings are assumed to have a uniform height of 1.
        min_radius (float, optional): Minimum distance multiplier used to exclude forces that would otherwise
                                        increase momentum. Forces with a distance below a threshold
    

## 1. Load data

Load the footprints geodataframe in `.shp` `.gpkg` `.geojson` format using `geopandas.read_file()`

Download an example footprints file if needed

In [ ]:
!curl -L -o footprints.gpkg https://github.com/GeomaticsCaminosUPM/seismic-building-exposure/raw/refs/heads/main/examples/footprint/footprints.gpkg

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  298k  100  298k    0     0   451k      0 --:--:-- --:--:-- --:--:--  451k


Load the footprints geodataframe in `.shp` `.gpkg` `.geojson` format using `geopandas.read_file()`

In [7]:
footprints_gdf = gpd.read_file('footprints.gpkg')

- All geometries should be single part and polygons.

    - Multi part geometries (MultiPolygon) would mean two buildings are contained in the same footprint geometry which should not happen.

    - Footprints must be polygons, not linestrings.

Check if all geometries are `Polygon`

In [8]:
if any(footprints_gdf.geometry.type == 'MultiPolygon'):
    print("There are multiplart geometries.")

if any(footprints_gdf.geometry.type.str.contains('Polygon') == False):
    print("Some geometries are not Polygon")

Explode multipart geometries into single parts if needed


Note: The row number will be reset. Maybe you do not know how to match the footprints with your data. An id column is a good idea to solve this issue.

In [9]:
footprints_gdf = footprints_gdf.explode().reset_index() # The new index column is the row number in the origninal dataframe
footprints_gdf

,index,id,city,height,n_storeys,year,code_quality,structural_system,simplified_structural_system,roof,geometry
0,0,1000009.0,san_jose,NaN,NaN,1975,pre_code,None,None,metallic,"POLYGON ((-84.10396 9.92614, -84.10405 9.92608..."
1,1,1219.0,san_jose,6.0,2.0,1975,pre_code,M,M,metallic,"POLYGON ((-84.10431 9.92621, -84.10426 9.92621..."
2,2,1000010.0,san_jose,NaN,NaN,1975,pre_code,None,None,dark_shadow,"POLYGON ((-84.10395 9.92618, -84.10397 9.92614..."
3,3,1221.0,san_jose,6.0,2.0,1985,low_code,M,M,bright_concrete,"POLYGON ((-84.10471 9.92629, -84.10472 9.92628..."
4,4,1222.0,san_jose,6.0,2.0,1988,medium_code,M,M,metallic,"POLYGON ((-84.10494 9.92626, -84.10494 9.92613..."
...,...,...,...,...,...,...,...,...,...,...,...
906,907,1000029.0,san_jose,NaN,NaN,1975,pre_code,None,None,metallic,"POLYGON ((-84.10486 9.93244, -84.10484 9.9325,..."
907,908,1000030.0,san_jose,NaN,NaN,1975,pre_code,None,None,dark_shadow,"POLYGON ((-84.10486 9.92836, -84.10487 9.9282,..."
908,909,1000031.0,san_jose,NaN,NaN,1985,low_code,None,None,metallic,"POLYGON ((-84.10393 9.92754, -84.10393 9.92762..."
909,910,1000032.0,san_jose,NaN,NaN,1975,pre_code,None,None,dark_shadow,"POLYGON ((-84.10333 9.92703, -84.10335 9.92701..."


## 2. Relative position


Determines if the building touches other structures (relative position in the city block). This is done by calculating "contact forces" that neighboring structures exert on the building. The force is proportional to the contact area (length of touching footprints multiplied by building height) in the normal direction of the touching plane.


Lets calculate the forces with `contact_forces_df()`

In [10]:
forces = seismicbuildingexposure.footprint.position.contact_forces_df(
    footprints_gdf,
    height_column=None, # Do not use any height column. All buildings have height 1.
    buffer=0.2,
    min_radius=0.5
) #20 cm of buffer
forces

,height,force,confinement_ratio,angular_acc,angle
0,1.0,0.697807,0.679394,0.320945,2.465851
1,1.0,0.641808,0.433961,1.613179,1.159877
2,1.0,0.341575,0.884813,0.001462,0.206058
3,1.0,0.020213,0.994469,0.067755,2.266547
4,1.0,0.287132,0.000000,0.220468,0.000000
...,...,...,...,...,...
906,1.0,0.014301,0.995964,0.046817,0.817430
907,NaN,0.000000,0.000000,0.000000,0.000000
908,1.0,1.114477,0.281350,0.209940,1.494991
909,1.0,1.401636,0.225779,0.923751,1.062266


Now determine the relative position using the forces calculated before with `relative_position()`

In [11]:
footprints_gdf['relative_position'] = seismicbuildingexposure.footprint.position.relative_position(
    forces,
    min_angular_acc=2.133,
    min_confinement=1,
    min_angle=0.78,
    min_force=0.1666
)
footprints_gdf

,index,id,city,height,n_storeys,year,code_quality,structural_system,simplified_structural_system,roof,geometry,relative_position
0,0,1000009.0,san_jose,NaN,NaN,1975,pre_code,None,None,metallic,"POLYGON ((-84.10396 9.92614, -84.10405 9.92608...",corner
1,1,1219.0,san_jose,6.0,2.0,1975,pre_code,M,M,metallic,"POLYGON ((-84.10431 9.92621, -84.10426 9.92621...",corner
2,2,1000010.0,san_jose,NaN,NaN,1975,pre_code,None,None,dark_shadow,"POLYGON ((-84.10395 9.92618, -84.10397 9.92614...",lateral
3,3,1221.0,san_jose,6.0,2.0,1985,low_code,M,M,bright_concrete,"POLYGON ((-84.10471 9.92629, -84.10472 9.92628...",isolated
4,4,1222.0,san_jose,6.0,2.0,1988,medium_code,M,M,metallic,"POLYGON ((-84.10494 9.92626, -84.10494 9.92613...",lateral
...,...,...,...,...,...,...,...,...,...,...,...,...
906,907,1000029.0,san_jose,NaN,NaN,1975,pre_code,None,None,metallic,"POLYGON ((-84.10486 9.93244, -84.10484 9.9325,...",isolated
907,908,1000030.0,san_jose,NaN,NaN,1975,pre_code,None,None,dark_shadow,"POLYGON ((-84.10486 9.92836, -84.10487 9.9282,...",isolated
908,909,1000031.0,san_jose,NaN,NaN,1985,low_code,None,None,metallic,"POLYGON ((-84.10393 9.92754, -84.10393 9.92762...",corner
909,910,1000032.0,san_jose,NaN,NaN,1975,pre_code,None,None,dark_shadow,"POLYGON ((-84.10333 9.92703, -84.10335 9.92701...",corner


We create a new column `"relative_position"` that classifies buildings into the following categories:
  1. **"torque"**: Buildings of class **confined** or **corner** with an angular acceleration exceeding the minimum.
  2. **"confined"**: Structures touching on both the left and right lateral sides.
  3. **"corner"**: Structures touching at a corner (determined by force and angle thresholds).
  4. **"lateral"**: Structures touching on either the left or right side.
  5. **"isolated"**: No touching structures.

## 3. Example code to plot results


Plot a map (needs pip install folium matplotlib mapclassify)

In [13]:
footprints_gdf.explore(
    column='relative_position',       # Column to base the color on
    cmap='gist_rainbow',        # Color map (Yellow to Red)
    legend=True ,           # Add a legend
)

## 4. Save results

In [ ]:
footprints_gdf.to_file('relative_position.gpkg')